# 06 Electrolyte (IA-ELY-001)

**Requirement:** `IA-ELY-001`

Patents: US12308414B2, EP4602674A1. Equations are paraphrased; see claims/paragraphs cited below.

## Governing equations


Default **6 M aqueous KOH**. Carbonation \(\mathrm{CO_2 + 2\,OH^- \to CO_3^{2-} + H_2O}\)
consumes OH- and may clog pores (US). Alternate recipes from IA-SYS-021 (LiOH blends,
NaOH, high hydroxide ≥ 7 M as a CE lever, EP). Conductivity from `ironair.properties`.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path
setup_path()
from plant_sim.params import default_params, CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G
from plant_sim.plotting import apply_style, timeseries, xy_plot, FIGURES
apply_style()
P = default_params()
print("6 M KOH default", P.c_KOH_mol_m3/1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G,2), round(CAPACITY_MAGNETITE_MAH_G,2))
print("figures ->", FIGURES)


## Isolated component simulation

In [ ]:
from plant_sim.components.electrolyte import Electrolyte
from plant_sim.components.base import integrate_component
ely = Electrolyte(P, filled=True)
t, y = integrate_component(lambda t,y: ely.rhs(t,y,{"r_iron_mol_s": 1e-6, "r_orr_mol_s": 5e-7, "x_CO2": 4.2e-4, "q_heat_W": 3.0, "T_amb_K": P.T_ep_sim_K}, {}), ely.y0(), (0, 2000), n_eval=150)
cM = y[0]/np.maximum(y[5],1e-12)/1000
timeseries(t/60, {"c_KOH_M": cM, "n_CO3": y[2], "T": y[4], "s_wet": y[6]}, xlabel="t (min)", ylabel="mixed SI", title="KOH, carbonate, T, wetting", callout="6 M KOH @ 303 K", stem="nb06_ely")
print("recipes", P.with_recipe("7M_KOH").c_KOH_mol_m3)


## Verification against patents / SSS002

In [ ]:
assert abs(P.c_KOH_mol_m3/1000 - 6) < 1e-9
print('PASS 6 M default')